# Model 2 V2: Fine-tuned DistilBERT — Longer Training (batch=64, 15 epochs)

**Architecture:** Same as V1 — DistilBERT [CLS] pooling + regression head  
**Change from V1:** batch_size=64, epochs=15, patience=3, warmup_steps=500  
**Hypothesis:** V1 val MAE was still decreasing at epoch 5 ($47.42) — more training should improve further

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

In [5]:
2

2

In [6]:
from pricer.items import Item
from pricer.distilbert_model_v2 import DistilBERTRunnerV2
from pricer.evaluator import evaluate, plot_training_history

## 1. Load Data

In [7]:
train, val, test = Item.from_hub("SeanSunny/items_full")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

Train: 800,000 | Val: 10,000 | Test: 10,000


## 2. Setup Model

Same DistilBERT architecture as V1, batch_size=64.

In [8]:
runner = DistilBERTRunnerV2(train, val[:1000])
runner.setup(batch_size=128)

Loading DistilBERT tokenizer...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Loading DistilBERT model...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

DistilBERT Regressor: 66,561,537 params (encoder: 66,362,880, head: 198,657)
Using cuda


## 3. Train

Max 15 epochs, early stopping patience=3, linear warmup 500 steps.

In [9]:
history = runner.train(epochs=15, patience=3, warmup_steps=1000)

Epoch 1/15:   0%|          | 0/6250 [00:00<?, ?it/s]

Epoch [1/15]
  Train Loss: 0.4730, Val Loss: 0.4041
  Val MAE: $56.03, LR: 0.00001887
  ** New best Val MAE: $56.03


Epoch 2/15:   0%|          | 0/6250 [00:00<?, ?it/s]

Epoch [2/15]
  Train Loss: 0.3877, Val Loss: 0.3899
  Val MAE: $53.59, LR: 0.00001752
  ** New best Val MAE: $53.59


Epoch 3/15:   0%|          | 0/6250 [00:00<?, ?it/s]

...

Epoch [13/15]
  Train Loss: 0.2227, Val Loss: 0.3370
  Val MAE: $45.28, LR: 0.00000270
  ** New best Val MAE: $45.28


Epoch 14/15:   0%|          | 0/6250 [00:00<?, ?it/s]

Epoch [14/15]
  Train Loss: 0.2179, Val Loss: 0.3392
  Val MAE: $45.75, LR: 0.00000135
  No improvement (1/3)


Epoch 15/15:   0%|          | 0/6250 [00:00<?, ?it/s]

Epoch [15/15]
  Train Loss: 0.2142, Val Loss: 0.3371
  Val MAE: $45.34, LR: 0.00000000
  No improvement (2/3)


## 4. Training History

In [10]:
plot_training_history(history, title="DistilBERT V2 (CLS, batch=64, 15 epochs)")

## 5. Evaluate on 200 Test Samples

In [11]:
evaluate(runner.inference, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$119 $121 $11 $6 $50 $106 $17 $44 $7 $6 $30 $147 $1 $9 $11 $2 $71 $21 $20 $75 $71 $85 $16 $177 $21 $197 $128 $2 $65 $57 $14 $12 $69 $1 $16 $210 $37 $30 $61 $19 $33 $41 $26 $193 $93 $3 $9 $4 $78 $14 $17 $47 $170 $25 $15 $38 $17 $97 $96 $6 $121 $47 $6 $55 $443 $6 $41 $317 $25 $21 $17 $3 $83 $7 $27 $1 $49 $2 $3 $1 $36 $19 $7 $50 $15 $76 $92 $94 $15 $6 $27 $7 $1 $5 $2 $28 $5 $37 $54 $149 $0 $25 $10 $10 $9 $296 $2 $321 $22 $58 $18 $36 $0 $39 $5 $33 $6 $2 $125 $176 $10 $18 $4 $5 $130 $23 $2 $25 $42 $57 $44 $17 $2 $3 $117 $0 $73 $59 $23 $1 $0 $58 $2 $18 $52 $13 $19 $283 $74 $6 $2 $24 $5 $5 $10 $39 $26 $0 $53 $7 $107 $10 $6 $1 $161 $2 $83 $27 $5 $1 $52 $12 $165 $26 $25 $6 $21 $21 $65 $9 $349 $7 $38 $22 $0 $22 $79 $9 $22 $4 $2 $1 $1 $136 $2 $22 $43 $2 $8 $11 

## 6. Save Model Weights

In [12]:
runner.save("distilbert_model_v2.pth")
print("Saved to distilbert_model_v2.pth")

Saved to distilbert_model_v2.pth


## 7. Sanity Check

In [13]:
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  ${sample.price:.2f}")
print(f"Predict: ${pred:.2f}")
print(f"Error:   ${abs(pred - sample.price):.2f}")

Product: Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal
Actual:  $219.00
Predict: $337.76
Error:   $118.76
